In [2]:
# ============================================================
#  RAG CHATBOT — Wikipedia + Groq + FAISS  (FIXED VERSION)
#  Author : Ali Abdal
#  Stack  : LangChain · FAISS · HuggingFace Embeddings · Groq
# ============================================================


# ─────────────────────────────────────────────
# CELL 1 — Install Dependencies (FIXED)
# ─────────────────────────────────────────────
!pip install -q \
    langchain==0.3.25 \
    langchain-community==0.3.23 \
    langchain-groq==0.3.2 \
    langchain-huggingface==0.1.2 \
    faiss-cpu==1.9.0 \
    sentence-transformers==3.4.1 \
    wikipedia==1.4.0 \
    gradio==5.29.1 \
    tiktoken==0.9.0

print("✅ All packages installed.")


# ─────────────────────────────────────────────
# CELL 2 — Set Groq API Key
# ─────────────────────────────────────────────
import os

# OPTION A: Colab Secrets (recommended)
# Left sidebar → 🔑 Secrets → Add "GROQ_API_KEY"
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Groq API key loaded from Colab Secrets.")
except Exception:
    # OPTION B: Paste key directly
    os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"
    print("⚠️  Paste your Groq key above. Get it free at console.groq.com")


# ─────────────────────────────────────────────
# CELL 3 — Imports (FIXED for LangChain 0.3.x)
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import wikipedia
import gradio as gr

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings          # FIXED
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain_core.documents import Document                     # FIXED

print("✅ All imports successful.")


# ─────────────────────────────────────────────
# CELL 4 — Configuration
# ─────────────────────────────────────────────
WIKIPEDIA_TOPICS = [
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Natural language processing",
    "Large language model",
    "Transformer (deep learning architecture)",
    "Retrieval-augmented generation",
    "Neural network",
]

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GROQ_MODEL      = "llama3-8b-8192"
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50

print("✅ Configuration set.")


# ─────────────────────────────────────────────
# CELL 5 — Load Wikipedia Articles
# ─────────────────────────────────────────────
def load_wikipedia_docs(topics):
    docs = []
    for topic in topics:
        try:
            page    = wikipedia.page(topic, auto_suggest=False)
            content = page.content[:8000]
            docs.append(Document(
                page_content=content,
                metadata={"source": page.url, "title": page.title}
            ))
            print(f"  ✅ {page.title}")
        except wikipedia.exceptions.DisambiguationError as e:
            try:
                page    = wikipedia.page(e.options[0], auto_suggest=False)
                content = page.content[:8000]
                docs.append(Document(
                    page_content=content,
                    metadata={"source": page.url, "title": page.title}
                ))
                print(f"  ✅ {page.title} (disambiguated)")
            except Exception:
                print(f"  ⚠️  Skipped: {topic}")
        except Exception as ex:
            print(f"  ❌ Failed: {topic} — {ex}")
    return docs

print("\n📥 Loading Wikipedia articles...")
raw_docs = load_wikipedia_docs(WIKIPEDIA_TOPICS)
print(f"\n✅ Articles loaded: {len(raw_docs)}")


# ─────────────────────────────────────────────
# CELL 6 — Split into Chunks
# ─────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators    = ["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_documents(raw_docs)
print(f"✅ Total chunks: {len(chunks)}")
print(f"   Preview: \"{chunks[0].page_content[:200]}...\"")


# ─────────────────────────────────────────────
# CELL 7 — Build FAISS Vector Store
# ─────────────────────────────────────────────
print("\n⚙️  Loading embedding model (~1 min first run)...")
embeddings = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)

print("⚙️  Building FAISS index...")
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("faiss_wiki_index")
print("✅ FAISS vector store built and saved.")


# ─────────────────────────────────────────────
# CELL 8 — RAG Chain Setup
# ─────────────────────────────────────────────
llm = ChatGroq(
    model      = GROQ_MODEL,
    temperature= 0.3,
    max_tokens = 1024,
)

retriever = vector_store.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 4}
)

memory = ConversationBufferMemory(
    memory_key      = "chat_history",
    return_messages = True,
    output_key      = "answer"
)

rag_chain = ConversationalRetrievalChain.from_llm(
    llm                    = llm,
    retriever              = retriever,
    memory                 = memory,
    return_source_documents= True,
    verbose                = False
)

print("✅ RAG chain ready — model:", GROQ_MODEL)


# ─────────────────────────────────────────────
# CELL 9 — Chat Logic
# ─────────────────────────────────────────────
def ask(question, chat_history):
    if not question.strip():
        return chat_history, "⚠️ Please type a question."
    try:
        result  = rag_chain.invoke({"question": question})
        answer  = result["answer"]
        sources = result.get("source_documents", [])

        seen, src_txt = set(), ""
        for doc in sources:
            url = doc.metadata.get("source", "")
            ttl = doc.metadata.get("title", "Source")
            if url and url not in seen:
                src_txt += f"• [{ttl}]({url})\n"
                seen.add(url)

        chat_history.append((question, answer))
        return chat_history, src_txt or "No sources found."
    except Exception as e:
        chat_history.append((question, f"❌ Error: {e}"))
        return chat_history, ""

def clear_all():
    memory.clear()
    return [], "", "🔄 Conversation cleared."


# ─────────────────────────────────────────────
# CELL 10 — Gradio UI
# ─────────────────────────────────────────────
CSS = """
#title    { text-align:center; font-size:1.8em; font-weight:bold; margin-bottom:4px; }
#subtitle { text-align:center; color:#888; font-size:0.95em; margin-bottom:16px; }
"""

with gr.Blocks(css=CSS, title="RAG Chatbot") as demo:

    gr.Markdown("## 🧠 RAG Chatbot — Wikipedia + Groq LLaMA 3", elem_id="title")
    gr.Markdown(
        "Powered by **LangChain · FAISS · HuggingFace Embeddings · Groq**  \n"
        "Ask anything about AI, ML, Deep Learning, NLP, Transformers, and more.",
        elem_id="subtitle"
    )
    gr.Markdown(
        "**📚 Topics:** " +
        " · ".join([f"`{t}`" for t in WIKIPEDIA_TOPICS])
    )
    gr.HTML("<hr>") # Changed from gr.Divider()

    with gr.Row():
        with gr.Column(scale=3):
            chatbot    = gr.Chatbot(label="Conversation", height=460, bubble_full_width=False)
            user_input = gr.Textbox(
                placeholder="e.g. What is Retrieval-Augmented Generation?",
                label="Your Question", lines=1
            )
            with gr.Row():
                submit_btn = gr.Button("Send 🚀", variant="primary")
                clear_btn  = gr.Button("🗑️ Clear", variant="secondary")
            status_box = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("### 📎 Sources")
            sources_box = gr.Markdown("_Sources appear here after each answer._")

    gr.Examples(
        examples=[
            ["What is Retrieval-Augmented Generation?"],
            ["How does a Transformer architecture work?"],
            ["Difference between machine learning and deep learning?"],
            ["How does FAISS do similarity search?"],
            ["What are large language models?"],
        ],
        inputs=user_input,
        label="💡 Example Questions"
    )

    gr.Markdown(
        "---\nBuilt by **Ali Abdal** · "
        "[LinkedIn](https://www.linkedin.com/in/ali-abdal-ml/) · "
        "[GitHub](https://github.com/aliabdal51214-cpu)"
    )

    state = gr.State([])

    submit_btn.click(ask, [user_input, state], [chatbot, sources_box]).then(
        lambda: "", outputs=[user_input]
    )
    user_input.submit(ask, [user_input, state], [chatbot, sources_box]).then(
        lambda: "", outputs=[user_input]
    )
    clear_btn.click(clear_all, outputs=[chatbot, sources_box, status_box])


# ─────────────────────────────────────────────
# CELL 11 — Launch App
# ─────────────────────────────────────────────
demo.launch(share=True, debug=False)
print("✅ App launched! Open the public Gradio link above ☝️")


# ─────────────────────────────────────────────
# CELL 12 — Download All Files to Your PC
# ─────────────────────────────────────────────
# Run this cell AFTER the app is working.
# It will download the notebook + requirements to your computer.

import json, textwrap
from google.colab import files

# ── 1. Save this script as a .py file ──
py_code = open(__file__).read() if hasattr(__builtins__, '__file__') else ""

# ── 2. Create requirements.txt ──
req = textwrap.dedent("""\
    langchain==0.3.25
    langchain-community==0.3.23
    langchain-groq==0.3.2
    langchain-huggingface==0.1.2
    faiss-cpu==1.9.0
    sentence-transformers==3.4.1
    wikipedia==1.4.0
    gradio==5.29.1
    tiktoken==0.9.0
""")
with open("requirements.txt", "w") as f:
    f.write(req)

# ── 3. Create .gitignore ──
gitignore = textwrap.dedent("""\
    faiss_wiki_index/
    __pycache__/
    *.pyc
    .env
    .DS_Store
""")
with open(".gitignore", "w") as f:
    f.write(gitignore)

# ── 4. Download files ──
print("📦 Downloading files to your PC...")
files.download("requirements.txt")
files.download(".gitignore")

# Download the notebook itself
import subprocess
nb_path = "/content/RAG_Chatbot_Wikipedia_Groq.ipynb"
# Convert current session to notebook via Colab's built-in mechanism
try:
    files.download(nb_path)
    print("✅ Notebook downloaded!")
except Exception:
    print("ℹ️  To download the notebook: File → Download → Download .ipynb")

print("\n✅ Done! Files downloaded:")
print("   • requirements.txt")
print("   • .gitignore")
print("   • RAG_Chatbot_Wikipedia_Groq.ipynb  (or use File → Download)")
print("\n📌 Upload these 3 files to your GitHub repo.")


✅ All packages installed.
⚠️  Paste your Groq key above. Get it free at console.groq.com
✅ All imports successful.
✅ Configuration set.

📥 Loading Wikipedia articles...
  ✅ Artificial intelligence
  ✅ Machine learning
  ✅ Deep learning
  ✅ Natural language processing
  ✅ Large language model
  ✅ Transformer (deep learning)
  ✅ Retrieval-augmented generation
  ✅ Neural network

✅ Articles loaded: 8
✅ Total chunks: 199
   Preview: "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and dec..."

⚙️  Loading embedding model (~1 min first run)...
⚙️  Building FAISS index...
✅ FAISS vector store built and saved.
✅ RAG chain ready — model: llama3-8b-8192
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58e553ecdd720026a9.gradio.live

This share link expires in 1 week. For free permane

✅ App launched! Open the public Gradio link above ☝️
📦 Downloading files to your PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ℹ️  To download the notebook: File → Download → Download .ipynb

✅ Done! Files downloaded:
   • requirements.txt
   • .gitignore
   • RAG_Chatbot_Wikipedia_Groq.ipynb  (or use File → Download)

📌 Upload these 3 files to your GitHub repo.
